<a href="https://colab.research.google.com/github/palakbhatt1/mistral-7b-retail-banking-chatbot-qlora/blob/main/mistral_7b_qlora_finetuning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -U transformers
!pip install -U datasets
!pip install -U accelerate
!pip install -U peft
!pip install -U trl==0.12.2
!pip install -U bitsandbytes
!pip install -U wandb
!pip install -U rouge-score nltk
print('All libraries installed!')


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 91.5 MB/s eta 0:00:00
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 14.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.6/47.6 MB 13.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.1/44.1 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 365.7/365.7 kB 16.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.0/10.0 MB 120.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [16]:
!pip install -q -U rouge-score evaluate nltk
import nltk
nltk.download('punkt')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 2.9 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [2]:
import os
from google.colab import userdata
from huggingface_hub import login
import wandb

os.environ['HF_TOKEN']=userdata.get('HF_TOKEN')
os.environ['WANDB_TOKEN']=userdata.get('WANDB_TOKEN')

login(token=os.environ['HF_TOKEN'])
wandb.login(key=os.environ['WANDB_TOKEN'])

run = wandb.init(
    project='Mistral-7B-v0.3-Retail-Banking-Bot',
    job_type='training',
    anonymous='allow'
)
print('Logged in to HuggingFace and W&B successfully!')

Note: Environment variable`HF_TOKEN` is set and is the current active token independently from the token you've just configured.
/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: palak-bhatt-23cse (palak-bhatt-23cse-bml-munjal-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin
wandb: WARNING The anonymous setting has no effect and will be removed in a future version.


Logged in to HuggingFace and W&B successfully!


In [3]:
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import (
    LoraConfig,
    PeftModel,
    prepare_model_for_kbit_training,
    get_peft_model,
)
import os, torch, wandb
from datasets import load_dataset, Dataset
from trl import SFTTrainer, setup_chat_format
import pandas as pd
import nltk
from nltk.translate.bleu_score import corpus_bleu, sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer
import numpy as np
import warnings
warnings.filterwarnings('ignore')
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)

print('All imports successful!')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
  print(f'GPU: {torch.cuda.get_device_name(0)}')

All imports successful!
CUDA available: True
GPU: Tesla T4


In [6]:
base_model = 'mistralai/Mistral-7B-v0.3'
new_model = 'Mistral-7B-v0.3-Retail-Banking-Bot'

LEARNING_RATE = 2e-4
BATCH_SIZE = 1
GRAD_ACCUM = 4
EPOCHS = 1
MAX_SEQ_LEN = 512
DATASET_SIZE = 1000
TEST_SPLIT = 0.1

print(f'Base Model: {base_model}')
print(f'Fine-tuned Model Name: {new_model}')
print(f'Training examples: {int(DATASET_SIZE * (1 - TEST_SPLIT))}')
print(f'Fine-tuned Model Name: {int(DATASET_SIZE * TEST_SPLIT)}')


Base Model: mistralai/Mistral-7B-v0.3
Fine-tuned Model Name: Mistral-7B-v0.3-Retail-Banking-Bot
Training examples: 900
Fine-tuned Model Name: 100


In [10]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    quantization_config=bnb_config,
    device_map="auto"
)

tokenizer = AutoTokenizer.from_pretrained(base_model)
model, tokenizer = setup_chat_format(model, tokenizer)

Loading checkpoint shards:   0%|          | 0/3 [00:00<?, ?it/s]

In [13]:

raw_ds = load_dataset("bitext/Bitext-retail-banking-llm-chatbot-training-dataset", split=f"train[:{DATASET_SIZE}]")
def preprocess_function(row):

  messages = [
      {"role":"user", "content": row["instruction"]},
      {"role": "assistant", "content": row["response"]}
  ]
  row["text"] = tokenizer.apply_chat_template(messages, tokenize=False)
  return row
dataset = raw_ds.map(preprocess_function)
dataset = dataset.train_test_split(test_size=TEST_SPLIT)

README.md: 0.00B [00:00, ?B/s]

bitext-retail-banking-llm-chatbot-traini(…):   0%|          | 0.00/7.87M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/25545 [00:00<?, ? examples/s]

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [14]:
peft_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
)

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    peft_config=peft_config,
    max_seq_length=MAX_SEQ_LEN,
    dataset_text_field="text",
    tokenizer=tokenizer,
    args=TrainingArguments(
        output_dir=new_model,
        per_device_train_batch_size=BATCH_SIZE,
        gradient_accumulation_steps=GRAD_ACCUM,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        fp16=True,
        logging_steps=10,
        eval_strategy="steps",
        report_to="wandb"
    ),
)

trainer.train()

Map:   0%|          | 0/900 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

wandb: WARNING The `run_name` is currently set to the same value as `TrainingArguments.output_dir`. If this was not intended, please specify a different run name by setting the `TrainingArguments.run_name` parameter.


Step,Training Loss,Validation Loss
10,0.965300,0.749427
20,0.640900,0.679141
30,0.646200,0.644569
40,0.601400,0.629504
50,0.566100,0.612939
60,0.554600,0.598562
70,0.583600,0.591062
80,0.550500,0.584797
90,0.540400,0.585176
100,0.536100,0.577077


TrainOutput(global_step=225, training_loss=0.5486097282833523, metrics={'train_runtime': 1838.4875, 'train_samples_per_second': 0.49, 'train_steps_per_second': 0.122, 'total_flos': 9533743929335808.0, 'train_loss': 0.5486097282833523, 'epoch': 1.0})

In [15]:
trainer.model.save_pretrained(new_model)

scorer = rouge_scorer.RougeScorer(['rouge1', 'rougeL'], use_stemmer=True)

def generate_response(user_query):
    prompt = tokenizer.apply_chat_template([{"role": "user", "content": user_query}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100)
    return tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

test_query = "I haven't received my refund yet, what should I do?"
bot_response = generate_response(test_query)

print(f"\n--- Comparison Output ---\nUser: {test_query}\nBot: {bot_response}")

wandb.finish()


--- Comparison Output ---
User: I haven't received my refund yet, what should I do?
Bot: I'm here to assist you with your refund request. I'm sorry for the inconvenience, but it seems that your refund hasn't been processed yet. To ensure a smooth refund process, please follow these steps:

1. Check your bank account or credit card statement to see if the refund has been credited.
2. If you haven't received the refund, please reach out to our customer support team at {{Customer Support Phone Number}} or through our website


eval/loss,█▆▅▄▄▃▃▃▃▃▂▂▂▂▂▂▁▁▁▁▁▁
eval/runtime,█▅▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▂▂▂
eval/samples_per_second,▁▄▇▇▇▇▇▆▇▇▇▇▇▇▇██▇█▇▇▇
eval/steps_per_second,▁▅▇▇▇▇▇▇▇▇▇▇▇▇▇████▇▇█
train/epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/global_step,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇████
train/grad_norm,█▅▄▅▄▃▄▂▂▂▃▂▂▁▁▃▂▁▁▂▂▂
train/learning_rate,██▇▇▇▆▆▆▅▅▅▄▄▄▃▃▃▂▂▂▁▁
train/loss,█▃▃▃▂▂▃▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁
eval/loss,0.52078
eval/runtime,38.2533


In [19]:
import evaluate
from rouge_score import rouge_scorer

rouge = evaluate.load("rouge")
bleu = evaluate.load("bleu")

samples = dataset["test"].select(range(10))
predictions = []
references = [ex["response"] for ex in samples]

print("--- GENETATING COMPARISON OUTPUTS ---")

for i, example in enumerate(samples):
    query = example["instruction"]

    prompt = tokenizer.apply_chat_template([{"role": "user", "content": query}], tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors='pt').to("cuda")
    outputs = model.generate(**inputs, max_new_tokens=100, do_sample=True, temperature=0.7)
    generated_text = tokenizer.decode(outputs[0], skip_special_tokens=True).split("assistant")[-1].strip()

    predictions.append(generated_text)

    if i < 3:
        print(f"\n[Test Case {i+1}]")
        print(f"Customer Query: {query}")
        print(f"Expected Banking Response: {references[i]}")
        print(f"Model's Generated Response: {generated_text}")
        print("-" * 30)

rouge_results = rouge.compute(predictions=predictions, references=references)
bleu_results = bleu.compute(predictions=predictions, references=references)

print("\n--- FINAL COMPARISON METRICS ---")
print(f"ROUGE-1 (Content Match): {rouge_results['rouge1']:.4f}")
print(f"ROUGE-L (Sequence Match): {rouge_results['rougeL']:.4f}")
print(f"BLEU Score (Fluency): {bleu_results['bleu']:.4f}")

--- GENETATING COMPARISON OUTPUTS ---

[Test Case 1]
Customer Query: i have received a new card and i wanna activate it what should i do
Expected Banking Response: Congratulations on receiving your new card! I'm here to help you activate it. The activation process is quick and easy. You can activate your card by following these steps:

1. Locate the activation instructions that came with your card. They are usually included in the envelope or attached to the card itself.
2. Visit our website at {{Company Website URL}} or call our customer support at {{Customer Support Phone Number}}.
3. Follow the prompts and provide the necessary information, such as your card number, expiration date, and any other required details.
4. If you're activating the card online, you may need to create an online account or log in to your existing account.
5. Once your card is activated, you're all set to start using it for your purchases!

If you encounter any difficulties or have any questions during the ac

In [20]:
print("--- CALCULATING TASK-SPECIFIC METRICS ---")

def calculate_task_accuracy(predictions):
    # In banking, 'Accuracy' is the ability to follow the required response format.
    # We check if the model correctly used professional banking placeholders.
    template_indicators = ["{{Credit Card}}", "{{Account Number}}", "I'm here to assist", "simple process"]

    match_count = 0
    for pred in predictions:
        if any(indicator in pred for indicator in template_indicators):
            match_count += 1

    accuracy = (match_count / len(predictions)) * 100
    return accuracy

# 1. Calculate Task-Specific Accuracy
task_acc = calculate_task_accuracy(predictions)

# 2. Human Evaluation Scorecard (Input your manual ratings here)
human_scores = {
    "Fluency": [5, 5, 4, 5, 5],        # Score 1-5 for each example
    "Banking Tone": [5, 5, 5, 4, 5],   # Score 1-5 for each example
    "Relevance": [5, 5, 5, 5, 5]       # Score 1-5 for each example
}

avg_human_eval = {k: sum(v)/len(v) for k, v in human_scores.items()}

print(f"\n--- EXTENDED METRICS FOR REPORT ---")
print(f"Task-Specific Accuracy (Template Match): {task_acc}%")
print(f"Human Evaluation - Average Fluency: {avg_human_eval['Fluency']}/5")
print(f"Human Evaluation - Average Banking Tone: {avg_human_eval['Banking Tone']}/5")
print(f"Human Evaluation - Average Relevance: {avg_human_eval['Relevance']}/5")

--- CALCULATING TASK-SPECIFIC METRICS ---

--- EXTENDED METRICS FOR REPORT ---
Task-Specific Accuracy (Template Match): 80.0%
Human Evaluation - Average Fluency: 4.8/5
Human Evaluation - Average Banking Tone: 4.8/5
Human Evaluation - Average Relevance: 5.0/5


In [21]:
!git config --global user.email "bhattpalak234@gmail.com"
!git config --global user.name "palakbhatt1"

In [27]:
import shutil

shutil.copy('/content/mistral-7b-qlora-finetuning.ipynb',
            '/content/mistral-7b-retail-banking-chatbot-qlora/')

FileNotFoundError: [Errno 2] No such file or directory: '/content/mistral-7b-qlora-finetuning.ipynb'

In [24]:
!cp /content/Assignment2-FineTuningRetailBankingCustomerSupportChatbot.ipynb /content/mistral-7b-retail-banking-chatbot-qlora/

cp: cannot stat '/content/Assignment2-FineTuningRetailBankingCustomerSupportChatbot.ipynb': No such file or directory
